# DeepFaune batch: starter analysis

This notebook reads the **per-shard CSVs** in your output directory directly (via
the `*.csv` glob), not the merged master. That keeps it decoupled from the merge
cadence and means it only ever reads complete files: the orchestrator writes each
shard CSV atomically (temp then rename), and this notebook skips `*.tmp` files and
`master.csv`. So it is safe to re-run repeatedly while the batch job is still
running. Just re-run all cells to refresh the totals.

In [ ]:
# --- configuration: edit these ---
OUT_DIR = "~/df_out"                 # directory holding the per-shard CSVs
PREDICTION_COL = "prediction_seq"    # or "prediction_image" for per-image labels
STATION_LEVELS = 2                   # trailing path parts used as the station key
PRIORITY_SPECIES = ["wolf", "bear", "lynx", "wild boar"]

In [ ]:
import glob
import os

import pandas as pd

CSV_COLUMNS = [
    "filename", "date", "seqnum",
    "prediction_seq", "score_seq",
    "prediction_image", "score_image",
    "animal_count", "human_count",
]


def station_from_filename(path, levels=STATION_LEVELS):
    """Derive a station key from an image path: its last `levels` directory parts."""
    directory = os.path.dirname(str(path)).replace("\\", "/")
    parts = [p for p in directory.split("/") if p]
    if not parts:
        return "(unknown)"
    return "/".join(parts[-levels:])


def load_shard_csvs(out_dir=OUT_DIR, station_levels=STATION_LEVELS):
    """Concatenate the per-shard CSVs in out_dir into one DataFrame.

    Skips master.csv and any .tmp file, so only complete shard outputs are read.
    A file that cannot be parsed yet is skipped and picked up on the next re-run.
    """
    out_dir = os.path.expanduser(out_dir)
    paths = sorted(glob.glob(os.path.join(out_dir, "*.csv")))
    paths = [p for p in paths if os.path.basename(p) != "master.csv"]
    frames = []
    for path in paths:
        try:
            frame = pd.read_csv(path)
        except Exception as exc:
            print(f"skipping {os.path.basename(path)}: {exc}")
            continue
        if frame.empty:
            continue
        frame["source_csv"] = os.path.basename(path)
        frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=CSV_COLUMNS + ["source_csv", "station"])
    data = pd.concat(frames, ignore_index=True)
    data["station"] = data["filename"].map(
        lambda x: station_from_filename(x, station_levels)
    )
    return data

In [ ]:
data = load_shard_csvs()
n_shards = data["source_csv"].nunique() if len(data) else 0
print(f"Loaded {n_shards} shard CSVs, {len(data)} rows from {os.path.expanduser(OUT_DIR)}")
data.head()

## Totals by predicted species

In [ ]:
if len(data):
    species_totals = data[PREDICTION_COL].value_counts()
    display(species_totals.to_frame("count"))
else:
    print("No shard CSVs found yet in", os.path.expanduser(OUT_DIR))

## Totals by station

In [ ]:
if len(data):
    station_totals = data.groupby("station").size().sort_values(ascending=False)
    display(station_totals.to_frame("count"))

## Priority species

Flags rows whose predicted label is one of `PRIORITY_SPECIES` (default wolf, bear,
lynx, wild boar). Edit the list in the configuration cell to change it.

In [ ]:
priority = {s.lower() for s in PRIORITY_SPECIES}
hits = data
if len(data):
    data["is_priority"] = data[PREDICTION_COL].astype(str).str.lower().isin(priority)
    hits = data[data["is_priority"]]
    n_stations = hits["station"].nunique() if len(hits) else 0
    print(f"{len(hits)} priority detections across {n_stations} stations")
    if len(hits):
        display(hits[PREDICTION_COL].value_counts().to_frame("count"))
        display(hits.groupby(["station", PREDICTION_COL]).size().to_frame("count"))

In [ ]:
review_cols = ["station", PREDICTION_COL, "score_seq", "date", "filename"]
if len(hits):
    display(
        hits.sort_values(["station", PREDICTION_COL])[review_cols].reset_index(drop=True)
    )
else:
    print("No priority-species detections yet")

Re-run all cells at any time to refresh while the batch job runs. For a single
shareable file, build the master with:

```
python deepfaune_batch.py --merge --out-dir <OUT_DIR>
```